## AI Math Assistant with Langchain Tool Calling

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from dotenv import load_dotenv
import re

In [ ]:
load_dotenv()

In [ ]:
#load the LLM
llm = ChatOpenAI(model="gpt-4", temperature=0.3, max_tokens=400)

In [ ]:
response = llm.invoke("What is tool calling?")

In [ ]:
print("\nResponse Content: ", response.content)

In [ ]:
# Function for adding numbers 
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]


    result = sum(numbers)
    return {"result": result}

In [ ]:
add_numbers("10 20 30")

In [ ]:
# Tool Class 
from langchain_core.tools import Tool
add_tool = Tool(
    name="AddTool",
    func=add_numbers,
    description="Adds a list of numbers and returns the result.")


In [ ]:
print("tool object", add_tool)

In [ ]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)

In [ ]:
print("Calling Tool Function:")
test_input = "10 20 40 b c"
print(add_tool.invoke(test_input))

In [ ]:
# @tool operator  -- Recommned way to create tools

@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    # Use regular expression to extract all numbers from the input 
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    
    result = sum(numbers)
    return {"result": result}


In [ ]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 

In [ ]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))

In [ ]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

In [ ]:
# Adding tool with two inputs : first add and second boolean as an input
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    
    if absolute:
        numbers = [abs(num) for num in numbers]
    
    return sum(numbers)


In [ ]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

In [ ]:
print(add_numbers_with_options.invoke({"numbers":[-2.1,-5.1,-3.0],"absolute":True}))
print(add_numbers_with_options.invoke({"numbers":[-2.1,-5.1,-3.0],"absolute":False}))

In [ ]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    
    if not matches:
        return {"result": "No numbers found in the input."}
    
    try:
        #numbers = [abs(float(num)) for num in matches]
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    
    except Exception as e:
        return {"result": f"Error processing numbers: {str(e)}"}

In [ ]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    
    return result

## create_react_agent
As LangChain's AgentExecutor is being deprecated, create_react_agent from LangGraph provides a more flexible and powerful alternative for building AI agents. This function creates a graph-based agent that works with chat models and supports tool-calling functionality.

## Key parameters of create_react_agent
1. model
The language model that powers the agent's reasoning.
Must support tool calling for full functionality.

2. tools
A list of tools the agent can use to perform actions.
Can be LangChain tools, Python functions with @tool decorator, or a ToolNode instance
Each tool should have a name, description, and implementation

3. prompt (optional):
Customizes the instructions given to the LLM
Can be:
A string (converted to a SystemMessage)
A SystemMessage object
A function that transforms the state
A Runnable that processes the state
and other parameters. To see more parameters, see docs.

## How it works
Unlike the legacy AgentExecutor, which used a fixed loop structure, create_react_agent creates a graph with these key nodes:

1. Agent Node: Calls the LLM with the message history
2. Tools Node: Executes any tool calls from the LLM's response
3. Continue/End Nodes: Manage the workflow based on whether tool calls are present

The graph follows this process:
1. User message enters the graph
2. LLM generates a response, potentially with tool calls
3. If tool calls exist, they're executed and their results are added to the message history
4. The updated messages are sent back to the LLM
5. This loop continues until the LLM responds without tool calls
6. The final state with all messages is returned

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_exec = create_react_agent(model=llm, tools=[sum_numbers_with_complex_output])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, 20, -30")]})

In [ ]:
print(msgs["messages"][-1].content)

In [ ]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in re.findall(r'-?\d+(?:\.\d+)?', inputs)]

    # If no numbers are found, return 0
    if not numbers:
      return {"result": 0}
    
    if "from" in inputs.lower() and len(numbers) >= 2:
      result = numbers[1] - numbers[0]
    
    else:
      result = numbers[0]
      for num in numbers[1:]:
        result -= num

    return {"result": result}

In [ ]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in re.findall(r'-?\d+(?:\.\d+)?', inputs)]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [ ]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in re.findall(r'-?\d+(?:\.\d+)?', inputs)]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

In [ ]:
tools = [add_numbers, subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent = create_react_agent(
    model=llm,
    tools=tools,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent",math_agent)

In [ ]:
# Test Cases
test_cases = [
    {
        "query": "Add 10, 50, and 70.",
        "expected": {"result": 130},
        "description": "Testing addition tool with sequential addition."
    },
    {
        "query": "Multiply 6, 3, and 2.",
        "expected": {"result": 36},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 200 by 5 and then by 2.",
        "expected": {"result": 20.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 20 from 50.",
        "expected": {"result": 30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [ ]:
def agent_decide_intent(query):
    prompt = f"""
    You are a math intent classifier.
    Decide the operation: add, subtract, multiply, divide.
    Return only one word.

    Query: {query}
    """
    return llm.invoke(prompt).content.strip().lower()


In [ ]:
correct_tasks = []

for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]

    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")

    # 🧠 AGENT decides intent
    intent = agent_decide_intent(query)
    print(f"Agent Intent: {intent}")

    # 📍 Deterministic routing
    if intent == "add":
        response = add_numbers(query)
    elif intent == "subtract":
        response = subtract_numbers(query)
    elif intent == "multiply":
        response = multiply_numbers(query)
    elif intent == "divide":
        response = divide_numbers(query)
    else:
        print("❌ Agent could not decide")
        continue

    tool_result = response["result"]

    print(f"Tool Result: {tool_result}")
    print(f"Expected Result: {expected_result}")

    if tool_result == expected_result:
        print(f"✅ Test Passed")
        correct_tasks.append(test["description"])
    else:
        print(f"❌ Test Failed")


In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia for actual information about a topic.
    
    Parameters: 
    - query (str): The topic or question to search for on Wikipedia.
    
    Returns:
    - str: A summary of relevant information from wikipedia
    """
    wikipedia = WikipediaAPIWrapper()
    return wikipedia.run(query)

In [99]:
search_wikipedia.invoke("What is tool calling?")

'Page: Cold calling\nSummary: Cold calling is the solicitation of business from potential customers who have had no prior contact with the salesperson conducting the call. It is an attempt to convince potential customers to purchase the salesperson\'s product or service.  Generally, it is an over-the-phone process, making it a form of telemarketing, but can also be done in-person by door-to-door salespeople.  Though cold calling can be used as a legitimate business tool, scammers can use cold calling as well.\n\nPage: WhatsApp\nSummary: WhatsApp Messenger, commonly known simply as WhatsApp, is an American social media, instant messaging (IM), and Voice over IP (VoIP) service accessible via desktop and mobile app. Owned by Meta Platforms, the service allows users to send text messages, voice messages, and video messages, make voice and video calls, and share images, documents, user locations, and other content. The service requires a cellular mobile telephone number to register. WhatsAp

In [100]:
tools_updated = [add_numbers, subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

In [101]:
# Create the agent with all the tools 
math_agent_updated = create_react_agent(
    model = llm,
    tools = tools_updated,
    prompt = "You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

In [102]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")
        


Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: What is the population of Canada? Multiply it by 0.75
Name: None

--- Message 2 ---
Type: AIMessage
Content: 
Name: None
Tool calls: [{'name': 'search_wikipedia', 'args': {'query': 'Population of Canada'}, 'id': 'call_FIQWFAP4jDmb9CHEK6cAaUmb', 'type': 'tool_call'}]

--- Message 3 ---
Type: ToolMessage
Content: Page: Population of Canada
Summary: Canada ranks 37th by population among countries of the world, comprising about 0.5% of the world's total, with about 41.5 million Canadians as of 2025. Despite being the second-largest country by total area (fourth-largest by land area), the vast majority of the country is sparsely inhabited, with most of its population south of the 55th parallel north. Just over 60 percent of Canadians live in just two provinces: Ontario and Quebec. Though Canada's overall population density is low, many regions in the south, such as the Quebec City–Windsor Corridor, have population densities h